In [1]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# Add highlighting specifications
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', style='lightskyblue')
pp.add_highlight(which='lower gap', style='bold yellow')

In [2]:
pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='hybrid', 
                                        num_hybrid_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v', op_iter_order=-2)\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\

combo_pool.print_library()

combo_pool: seq_length=None, num_states=40
state  name             seq
    0  mut_0.v0         TCCCGACT<cre>GGAAAGCaGGCAaTGAGCACcaAGG</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1         TCCCGACT<cre>GGAAAGCaGGCAaTGAGCACcaAGG</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_1.v0         TCCCGACT<cre>GGAAAcCGGGCAGTGAGgACACAta</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v1         TCCCGACT<cre>GGAAAcCGGGCAGTGAGgACACAta</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_2.v0         TCCCGACT<cre>GGAAAGaGGGCAGaGAGCACAatGG</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_2.v1         TCCCGACT<cre>GGAAAGaGGGCAGaGAGCACAatGG</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_3.v0         TCCCGACT<cre>GGAAcGCGGGCAGTctaCACACAGG</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_3.v1         TCCCGACT<cre>GGAAcGCGGGCAGTctaCACACAGG</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_4.v0         TCCCGACT<cre>GcAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_4.v1         TCCCGACT<cre>GcAAAGCGGG

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [8]:
df = combo_pool.generate_library(report_design_cards=True)
renamed_cols = {
    'name': 'name',
    'op[9]:repeat.state': 'repeat',
    #'pool[8].state': 'cre',
    'op[8]:stack.state': 'branch_state',
    'op[1]:mutagenize.state': 'mut_state',
    'op[2]:deletion_scan(marker_scan).state': 'del_state',
    'insertion_pool.state': 'ins_state',
    'op[6]:insertion_scan(marker_scan).state': 'ins_pos_state',
    'op[4]:from_seqs.state': 'ins_site_state',
    
}
tmp_df = df.rename(columns=renamed_cols)[renamed_cols.values()]
display(tmp_df)

,name,repeat,branch_state,mut_state,del_state,ins_state,ins_pos_state,ins_site_state
0,mut_0.v0,0,0,0,None,None,None,None
1,mut_0.v1,1,0,0,None,None,None,None
2,mut_1.v0,0,0,1,None,None,None,None
3,mut_1.v1,1,0,1,None,None,None,None
4,mut_2.v0,0,0,2,None,None,None,None
5,mut_2.v1,1,0,2,None,None,None,None
6,mut_3.v0,0,0,3,None,None,None,None
7,mut_3.v1,1,0,3,None,None,None,None
8,mut_4.v0,0,0,4,None,None,None,None
9,mut_4.v1,1,0,4,None,None,None,None


In [ ]:
combo_pool.counter.print_dag()

In [ ]:
df = combo_pool.generate_library(report_design_cards=True)
state_cols = [col for col in df.columns if '.state' in col]
print('\n'.join(state_cols))

In [ ]:
renamed_cols = {
    'name': 'name',
    'op[10]:get_kmers.key.kmer': 'bc_seq',
    'op[9]:repeat.state': 'v',
    'op[1]:mutagenize.key.positions': 'mut_positions',
    'op[1]:mutagenize.key.wt_chars': 'mut_wt_chars',
    'op[1]:mutagenize.key.mut_chars': 'mut_mut_chars',
    'op[2]:deletion_scan(marker_scan).key.start': 'del_start',
    'op[2]:deletion_scan(marker_scan).key.stop': 'del_stop',
    #'op[2]:deletion_scan(marker_scan).key.length': 'del_length',
    'op[2]:deletion_scan(marker_scan).key.region_content': 'deleted_seq',
    'op[6]:insertion_scan(marker_scan).key.start': 'ins_start',
    'op[6]:insertion_scan(marker_scan).key.stop': 'ins_stop',
    #'op[6]:insertion_scan(marker_scan).key.length': 'ins_length',
    'op[6]:insertion_scan(marker_scan).key.strand': 'ins_strand',
    'op[6]:insertion_scan(marker_scan).key.region_content': 'replaced_seq',
    'op[4]:from_seqs.key.seq_name': 'site_name',
    'sites_pool.seq': 'site_seq',
}
tmp_df = df.rename(columns=renamed_cols)[renamed_cols.values()]
display(tmp_df)